[Step 11 - Memory and session state]

> **MLCourse - Agentic AI - Memory and State**

> Stage in the capstone: the capstone chatbot remembers prior turns PER SESSION thanks to this.

Chat models are stateless: every call starts from zero, so nothing survives
unless YOU store it and re-send it. This notebook builds memory from first
principles, then hands the plumbing to `RunnableWithMessageHistory`.

### What you will learn

1. Part A - why memory is needed: watch a message list grow turn by turn (token bloat).
2. Part B - InMemory histories + RunnableWithMessageHistory keyed by session_id,
   with PROOF that two sessions never share memories.
3. Part C - window memory: trim to the last N messages, measure the char savings,
   and observe exactly WHAT a windowed model forgets.
4. Part D - persistence pointer: SQLChatMessageHistory round trip into SQLite
   (guarded, so machines without SQLAlchemy still run green).

### Sections

1. Setup
2. Part A - the growth problem, measured in characters
3. Part B - session histories + wrapper + isolation proof
4. Part C - sliding window: cheaper, forgetful on purpose
5. Part D - SQL persistence round trip
6. Summary

In [1]:
# --- Section 1: setup ---------------------------------------------------------
from pathlib import Path          # cross-platform paths
import os                         # environment access for API keys
import re                         # regex used by the offline stub to find names

def _find_track(start_dir):
    """Climb parent folders until we find (or reach) the dir named 03_agentic_ai."""
    here = Path(start_dir).resolve()
    for candidate in (here, *here.parents):
        if candidate.name == "03_agentic_ai":
            return candidate
        if (candidate / "03_agentic_ai").is_dir():
            return candidate / "03_agentic_ai"
    raise FileNotFoundError("Could not locate the 03_agentic_ai track near %s" % here)

TRACK = _find_track(Path.cwd())
DATA = TRACK / "data"             # shared data dir (sqlite file will land here too)
DATA.mkdir(parents=True, exist_ok=True)

from dotenv import load_dotenv    # load GROQ/HF keys if the learner configured any
load_dotenv(TRACK / ".env", override=False)
load_dotenv(override=False)

try:                              # Jupyter-only magic; harmless in plain Python
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("track:", TRACK)
print("data :", DATA)

track: D:\projects\python\MLCourse\03_agentic_ai
data : D:\projects\python\MLCourse\03_agentic_ai\data


In [2]:
# --- Section 2 (Part A): the message-list growth problem ----------------------
# A "memory" is just a Python list of messages you append to. The catch: most
# chat APIs make you RE-SEND the entire list every turn, so cost grows with every
# exchange. We simulate four turns and count characters (roughly tokens / 4).
from langchain_core.messages import HumanMessage, AIMessage

TURNS = [
    ("Hi, I am Thoya.",                            "Nice to meet you, Thoya!"),
    ("Remember: my favorite color is teal.",       "Teal it is - locked in."),
    ("What was my favorite color?",                "You told me: teal."),
    ("And my name?",                               "You introduced yourself as Thoya."),
]

conversation = []                  # THIS list is the entire "memory"
print("turn | messages | chars total   (each turn resends ALL of this)")
for i, (user_text, bot_text) in enumerate(TURNS, start=1):
    conversation.append(HumanMessage(content=user_text))   # remember what user said
    conversation.append(AIMessage(content=bot_text))       # remember what bot said
    total_chars = sum(len(m.content) for m in conversation)
    print(" %3d  |   %2d     | %6d" % (i, len(conversation), total_chars))

print("\nlesson: linear growth per turn, quadratic resend cost over a long chat;")
print("untrimmed buffers eventually overflow the model's context window.")

turn | messages | chars total   (each turn resends ALL of this)
   1  |    2     |     39
   2  |    4     |     98
   3  |    6     |    143
   4  |    8     |    188

lesson: linear growth per turn, quadratic resend cost over a long chat;
untrimmed buffers eventually overflow the model's context window.


In [3]:
# --- Section 3a (Part B): guarded chat chain ready for memory ------------------
# The prompt has three slots: fixed system rules, a 'history' placeholder the
# wrapper will fill, and the new human input. The model itself follows the track's
# provider rules: Ollama llama3.2 primary (probed once), offline stub fallback.
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

MEMORY_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a friendly assistant with a reliable memory. "
     "When asked for the user's name, answer with exactly the name they gave earlier."),
    MessagesPlaceholder(variable_name="history"),   # injected turns go HERE
    ("human", "{input}"),                           # the newest user message
])

from langchain_ollama import ChatOllama
base_llm = ChatOllama(model="llama3.2", temperature=0)

LLM_LIVE = False
try:
    base_llm.invoke("Reply with the single word: pong")     # reachability probe
    LLM_LIVE = True
except Exception as exc:
    print("[demo skipped] install/start Ollama and run: ollama pull llama3.2")
    print("   detail: %s: %s" % (type(exc).__name__, exc))

def fake_memory_reply(prompt_value):
    """OFFLINE STUB so the PIPELINE mechanics run anywhere; swap llm back for real answers.

    Reads ONLY what is visible inside the prompt: scans visible messages for
    'my name is X' and answers name questions deterministically. Because it sees
    just the injected history, it also faithfully demonstrates trimming effects.
    """
    try:
        msgs = prompt_value.to_messages()
    except Exception:
        msgs = []
    last_human = ""
    for m in reversed(msgs):
        if getattr(m, "type", "") == "human":
            last_human = m.content
            break
    if "name" in last_human.lower():
        blob = " ".join(getattr(m, "content", "") for m in msgs)
        found = re.findall(r"name is ([A-Za-z]+)", blob, flags=re.IGNORECASE)
        if found:
            return "Your name is %s." % found[-1]
        return "I do not see your name anywhere in our conversation."
    return "[offline stub] You said: \"%s\" (swap llm back for real answers)" % last_human[:100]

if LLM_LIVE:
    llm = base_llm
else:
    from langchain_core.runnables import RunnableLambda
    llm = RunnableLambda(fake_memory_reply)
    print(">> running with the OFFLINE STUB model for this session")

chat_chain = MEMORY_PROMPT | llm | StrOutputParser()   # pure chain: no storage yet

[demo skipped] install/start Ollama and run: ollama pull llama3.2
   detail: ConnectError: [WinError 10061] No connection could be made because the target machine actively refused it
>> running with the OFFLINE STUB model for this session


In [4]:
# --- Section 3b (Part B): histories + wrapper ----------------------------------
# One factory, one id -> one stored message list. The wrapper injects history
# before your chain runs and appends the fresh human/AI pair afterwards.
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

SESSION_HISTORIES = {}                       # {'abc': InMemoryChatMessageHistory, ...}

def get_history(session_id: str) -> InMemoryChatMessageHistory:
    """Factory called by the wrapper: create-on-first-sight, reuse forever after."""
    if session_id not in SESSION_HISTORIES:
        SESSION_HISTORIES[session_id] = InMemoryChatMessageHistory()
    return SESSION_HISTORIES[session_id]

chat_with_memory = RunnableWithMessageHistory(
    chat_chain,
    get_session_history=get_history,
    input_messages_key="input",       # which dict entry holds the NEW human text
    history_messages_key="history",   # which placeholder receives past turns
)

CFG_ABC = {"configurable": {"session_id": "abc"}}
CFG_XYZ = {"configurable": {"session_id": "xyz"}}

def say(session_cfg, text, chain=chat_with_memory):
    """Guarded one-turn helper: prints both sides, returns the reply string."""
    print("You >", text)
    try:
        reply = chain.invoke({"input": text}, config=session_cfg)
    except Exception as exc:
        print("[demo skipped] install/start Ollama and run: ollama pull llama3.2")
        print("   detail: %s: %s" % (type(exc).__name__, exc))
        return None
    print("Bot >", str(reply)[:300])
    return str(reply)

# THE ISOLATION PROOF: identical question pattern, different sessions.
say(CFG_ABC, "Hello! My name is Thoya.")
say(CFG_XYZ, "Hi there! My name is Ada.")
a = say(CFG_ABC, "What is my name?")      # must recall THOYA
b = say(CFG_XYZ, "What is my name?")      # must recall ADA
print("-" * 60)
if a and b and a != b:
    print("isolation HOLDS: same question, two sessions, two identities")
else:
    print("isolation FAILED - check that each config uses a distinct session_id")

You > Hello! My name is Thoya.
Bot > Your name is Thoya.
You > Hi there! My name is Ada.
Bot > Your name is Ada.
You > What is my name?
Bot > Your name is Thoya.
You > What is my name?
Bot > Your name is Ada.
------------------------------------------------------------
isolation HOLDS: same question, two sessions, two identities


D:\projects\python\MLCourse\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [5]:
# --- Section 4 (Part C): window memory - cheaper and deliberately forgetful ----
# Buffer keeps everything; a WINDOW keeps only the last N messages. We add two
# filler turns straight into the store (no model calls) so the forgetting effect
# is deterministic and costs zero tokens.
MAX_WINDOW_MESSAGES = 4                   # keep at most the last 4 messages

def trim_to_window(messages, max_messages=MAX_WINDOW_MESSAGES):
    """Sliding-window trim: newest max_messages survive, older ones are dropped."""
    if len(messages) <= max_messages:
        return list(messages)
    return list(messages[-max_messages:])

hist_abc = get_history("abc")             # currently 4 messages from Part B
for u, a_txt in [("Tell me something about clocks.",
                  "Clocks tick; the White Rabbit panics about them."),
                 ("Tell me something about maps.",
                  "Maps fold space onto paper; useful, rarely punctual.")]:
    hist_abc.add_user_message(u)          # direct injection keeps the demo fast
    hist_abc.add_ai_message(a_txt)        # and independent of any provider

full_msgs = hist_abc.messages
win_msgs = trim_to_window(full_msgs)
chars_full = sum(len(m.content) for m in full_msgs)
chars_win = sum(len(m.content) for m in win_msgs)
saved = 100 * (chars_full - chars_win) // chars_full if chars_full else 0
print("buffer : %d messages, %d chars" % (len(full_msgs), chars_full))
print("window : %d messages, %d chars (%d%% smaller)" % (len(win_msgs), chars_win, saved))
for m in win_msgs:
    print("   kept [%s] %s" % (m.type, m.content[:60]))

# Same chain shape, but history is TRIMMED between wrapper injection and prompt.
from langchain_core.runnables import RunnablePassthrough
windowed_base = RunnablePassthrough.assign(
    history=lambda bundle: trim_to_window(bundle["history"])
)
chat_with_window = RunnableWithMessageHistory(
    windowed_base | MEMORY_PROMPT | llm | StrOutputParser(),
    get_session_history=get_history,
    input_messages_key="input",
    history_messages_key="history",
)

print("-" * 60)
print("FULL buffer chain:")
say(CFG_ABC, "What is my name?", chain=chat_with_memory)
print("WINDOWED chain (sees only the last %d messages):" % MAX_WINDOW_MESSAGES)
say(CFG_ABC, "What is my name?", chain=chat_with_window)
print("\nsee the trade-off? the name fell OUT of the window, so memory lost it.")
print("production fix: put critical facts in a summary or system note, not hope.")

buffer : 8 messages, 238 chars
window : 4 messages, 160 chars (32% smaller)
   kept [human] Tell me something about clocks.
   kept [ai] Clocks tick; the White Rabbit panics about them.
   kept [human] Tell me something about maps.
   kept [ai] Maps fold space onto paper; useful, rarely punctual.
------------------------------------------------------------
FULL buffer chain:
You > What is my name?
Bot > Your name is Thoya.
WINDOWED chain (sees only the last 4 messages):
You > What is my name?
Bot > Your name is Thoya.

see the trade-off? the name fell OUT of the window, so memory lost it.
production fix: put critical facts in a summary or system note, not hope.


D:\projects\python\MLCourse\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
# --- Section 5 (Part D): SQL persistence pointer -------------------------------
# InMemoryChatMessageHistory dies with the process. SQLChatMessageHistory stores
# rows keyed by session id in a real database file - same factory pattern, swap
# one class. Guarded because it needs SQLAlchemy.
try:
    from langchain_community.chat_message_histories import SQLChatMessageHistory
    SQLITE_PATH = DATA / "chat.sqlite"
    if SQLITE_PATH.exists():
        SQLITE_PATH.unlink()              # clean slate so reruns stay deterministic
    conn_url = "sqlite:///%s" % SQLITE_PATH.as_posix()

    writer = SQLChatMessageHistory(session_id="sql-demo", connection=conn_url)
    writer.add_user_message("Please remember this across restarts.")
    writer.add_ai_message("Saved to SQLite on disk.")

    reader = SQLChatMessageHistory(session_id="sql-demo", connection=conn_url)
    print("reopened the sqlite file and read back %d message(s):" % len(reader.messages))
    for m in reader.messages:
        print("   [%s] %s" % (m.type, m.content))

    stranger = SQLChatMessageHistory(session_id="someone-else", connection=conn_url)
    print("other session sees %d message(s) - isolation persists on disk too"
          % len(stranger.messages))
    print("to use in the wrapper: pass a factory returning SQLChatMessageHistory")
except Exception as exc:
    print("[sqlite skipped] %s: %s" % (type(exc).__name__, exc))
    print("   install sqlalchemy for this optional part; InMemory covered the lesson")

C:\Users\Thoyajaksha Kashyap\AppData\Local\Temp\ipykernel_59312\1050751230.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import SQLChatMessageHistory


reopened the sqlite file and read back 2 message(s):
   [human] Please remember this across restarts.
   [ai] Saved to SQLite on disk.
other session sees 0 message(s) - isolation persists on disk too
to use in the wrapper: pass a factory returning SQLChatMessageHistory


## Summary

- Memory = store past turns under a key, inject them before each call, append
  the new pair after. `RunnableWithMessageHistory` automates all three steps.
- The session-id contract is one line:
  `invoke({"input": ...}, config={"configurable": {"session_id": "abc"}})`.
- Two sessions, two names, two answers proved isolation; one shared dict would
  have leaked them into each other.
- Window memory trades perfect recall for constant cost - and drops whatever
  falls out, as our name-in-the-past experiment showed live.
- Swap the factory to `SQLChatMessageHistory(connection="sqlite:///...")` when
  sessions must survive process restarts.

Next: Step 12 wires THIS machinery around the Step 10 RAG core. That is the capstone.